# Script to test trained Collision Avoidance Maneuver (CAM) policy



In [ ]:

import os
import sys
import random 
import json 

import numpy as np
import pandas as pd
import torch as T

from leo_gym.utils.utils import seed_all, create_dir
from pathlib import Path


In [ ]:
from leo_gym.gyms.cam_gym import CamEnv, CamEnvConfig
# from train_cam_hppo_cfg import env_cfg


flag = 0
SEED = np.random.randint(0,100000)
SEED = 63289
print(SEED)
# 37255
# SEED = 62008
# SEED = 55814
# SEED = 49497
# SEED = 84606

# 80366
# 62266
# 33962
# 37950
# SEED = 57761
# 63289


env_cfg = "/home/nektaf/workspace_optacom/libs/leo_gym/notebooks/Collision_Avoidance_Maneuvers/trained_policy_net_cfg/env_cfg.json"
def make_env():
    return CamEnv(env_cfg, seed=SEED)

num_envs = 1
env = make_env()
obs = env.reset()


In [ ]:

from leo_gym.rl_algorithms.h_ppo.h_ppo_agent import Agent


policy_file_path = "/home/nektaf/workspace_optacom/libs/leo_gym/notebooks/Collision_Avoidance_Maneuvers/trained_policy_net_cfg/policynet.pth"
critic_file_path = "/home/nektaf/workspace_optacom/libs/leo_gym/notebooks/Collision_Avoidance_Maneuvers/trained_policy_net_cfg/valuenet.pth"
ppo_config = "/home/nektaf/workspace_optacom/libs/leo_gym/notebooks/Collision_Avoidance_Maneuvers/trained_policy_net_cfg/ppo_cfg.json"

ppo = Agent(
    env_obs=env.observation_space,
    env_actions=env.action_space,
    ppo_cfg=ppo_config,
    train=False,
    policy_file_path=policy_file_path,
    critic_file_path=critic_file_path,
    device='cpu'
)


In [ ]:
truncated = False 
terminated = False
state = env.reset()[0]

rewards =[]
costs= []
simulation_times = []

# env.reset()

for i in range(100):
    
    action_dis, action_cont, prob_dis, prob_cont, val = ppo.choose_action(state, deterministic=True)
            
    action = {
            "discrete": action_dis,                     
            "continuous": action_cont,
        }
    
    next_state, reward, terminated, truncated, info = env.step(action)

    state = next_state
    DebrisSwarm_1 = env.DebrisSwarm_1
    rewards.append(reward)
    costs.append(info["cost"])
    print(action)
    simulation_times.append(info["n"])

    if terminated: 
        break
    if truncated:
        break
    
print(i)

In [ ]:
sum(rewards)

In [ ]:
env.plot_states_interactive()

In [ ]:
import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

for i, states in enumerate(DebrisSwarm_1.primary_sat_and_debris_rvm):
    states = np.array(states)

    x = states[DebrisSwarm_1.conjuction_time-10:DebrisSwarm_1.conjuction_time+10, 0]
    y = states[DebrisSwarm_1.conjuction_time-10:DebrisSwarm_1.conjuction_time+10, 1]
    z = states[DebrisSwarm_1.conjuction_time-10:DebrisSwarm_1.conjuction_time+10, 2]
    
    if i != 0:
        color = "red"
    else:
        color = None

    fig.add_trace(go.Scatter3d(x=x, y=y, z=z,  marker=dict(color=color), name=f'Orbit {i+1}'))

    

radius = 2.5e5

conjuction_point=DebrisSwarm_1.primary_sat_and_debris_rvm[0][DebrisSwarm_1.conjuction_time]

x = conjuction_point[0]
y = conjuction_point[1]
z = conjuction_point[2]

phi, theta = np.mgrid[0:2 * np.pi:30j, 0:np.pi:20j]

xx = radius*np.cos(theta)*np.sin(phi) + x
yy = radius*np.sin(theta)*np.sin(phi) + y
zz = radius*np.cos(phi) + z

fig.add_trace(go.Surface(x=xx, y=yy, z=zz, opacity=0.4, showscale=False, colorscale=[[0, 'gray'], [1, 'gray']]))

fig.update_layout(scene=dict(
                    xaxis_title='X',
                    yaxis_title='Y',
                    zaxis_title='Z',
                    aspectmode='data'),
                    width=800*1.2, height=800)

fig.update_layout(scene_camera=dict(eye=dict(x=-1.25, y=-1.25, z=0.7)))

fig.show()


In [ ]:
DebrisSwarm_1.plot_projected_position_bplane()

In [ ]:
# %matplotlib widget
env.publication_ready_plots(save_path=os.getcwd())

In [ ]:
# os.makedirs("temp", exist_ok=True)
# path = create_dir(parent_name="temp/test_run")
# path = os.path.abspath(path)
# os.chdir(path)

# DebrisSwarm_1.save_states(path=path)

# config=[config_path,actor_file_path,critic_file_path,SEED]

# with open('config_paths.txt', 'w') as f:
#     for path in config:
#         f.write(f"{path}\n")


In [ ]:
print(os.getcwd())